<a href="https://colab.research.google.com/github/Hackathon-05-06-2026/Hackathon_Files/blob/main/Group_R_preprocess_test_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ==========================================
# 1. MOUNT DRIVE & IMPORT LIBRARIES
# ==========================================
from google.colab import drive
import pandas as pd
import numpy as np
import glob
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler

print("🔑 Step 1: Connecting to Google Drive...")
drive.mount('/content/drive')

# ==========================================
# 2. LOAD FILES FROM YOUR SHARED DRIVE
# ==========================================
print("\n🔍 Step 2: Locating training super-dataset and raw test dataset...")
search_train = '/content/drive/MyDrive/*Dataset*C*/super_dataset_C_train.csv'
search_test = '/content/drive/MyDrive/*Dataset*C*/dataset_C_testing.csv'

# Check fallback path patterns just in case
train_files = glob.glob(search_train) or glob.glob('/content/drive/MyDrive/Hackathon_05*06/super_dataset_C_train.csv')
test_files = glob.glob(search_test) or glob.glob('/content/drive/MyDrive/Hackathon_05*06/dataset_C_testing.csv')

if len(train_files) > 0 and len(test_files) > 0:
    df_super_train = pd.read_csv(train_files[0])
    df_raw_test = pd.read_csv(test_files[0])
    print(f"🎯 SUCCESS! Referenced Training Columns from: {train_files[0]}")
    print(f"🎯 SUCCESS! Loaded Raw Test Data from: {test_files[0]}")
else:
    raise FileNotFoundError("❌ ERROR: Could not locate the required dataset files in your shared Drive folder.")

# Extract the respondent_id column to attach back at the very end
test_ids = df_raw_test[['respondent_id']].copy()

# ==========================================
# 3. APPLY ALIGNED PREPROCESSING PIPELINE
# ==========================================
print("\n⚙️ Step 3: Structuring test data to match the training feature grid...")

# Drop ID column to isolate raw variables
X_test_raw = df_raw_test.drop(columns=['respondent_id'])

# Separate continuous numeric columns from object strings
categorical_cols = X_test_raw.select_dtypes(include=['object']).columns.tolist()
numeric_cols = X_test_raw.select_dtypes(include=['int64', 'float64']).columns.tolist()

# 1. Handle Categorical Missing Values & One-Hot Encode
cat_imputer = SimpleImputer(strategy='constant', fill_value='Missing')
X_test_cat_imputed = pd.DataFrame(cat_imputer.fit_transform(X_test_raw[categorical_cols]), columns=categorical_cols)
X_test_cat_encoded = pd.get_dummies(X_test_cat_imputed, drop_first=True)

# 2. Recombine numerical columns with dummy columns
X_test_combined = pd.concat([X_test_raw[numeric_cols], X_test_cat_encoded], axis=1)

# 3. FORCE MATRIX COLUMN ALIGNMENT (CRITICAL STEP)
# Get the exact feature array shape from the training super-dataset (dropping target)
X_train_features = df_super_train.drop(columns=['covid_vaccine']).columns

# reindex ensures the test matrix has the exact same columns in the exact same sequence.
# If a dummy category isn't present in the test set, it safely populates it with 0s.
X_test_aligned = X_test_combined.reindex(columns=X_train_features, fill_value=0)

# 4. Impute any remaining continuous numerical blanks via KNN
knn_imputer = KNNImputer(n_neighbors=5)
X_test_imputed = pd.DataFrame(knn_imputer.fit_transform(X_test_aligned), columns=X_test_aligned.columns)

# 5. Apply the standard feature scaler
scaler = StandardScaler()
X_test_scaled = pd.DataFrame(scaler.fit_transform(X_test_imputed), columns=X_test_imputed.columns)

# 6. Bind the tracking respondent_id column back to the front
df_super_test = pd.concat([test_ids, X_test_scaled], axis=1)

# ==========================================
# 4. EXPORT COMPLETED TEST FILE
# ==========================================
test_output_path = test_files[0].replace('dataset_C_testing.csv', 'super_dataset_C_test.csv')
df_super_test.to_csv(test_output_path, index=False)

print("\n================ 🏆 PROCESSING COMPLETE ================")
print(f"💾 Transformed testing dataset saved safely at:\n   {test_output_path}")
print(f"📊 Final Test Dimensions: {df_super_test.shape[0]} samples | {df_super_test.shape[1]} features (including respondent_id)")

🔑 Step 1: Connecting to Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

🔍 Step 2: Locating training super-dataset and raw test dataset...
🎯 SUCCESS! Referenced Training Columns from: /content/drive/MyDrive/Hackathon_05 06/super_dataset_C_train.csv
🎯 SUCCESS! Loaded Raw Test Data from: /content/drive/MyDrive/Hackathon_05 06/dataset_C_testing.csv

⚙️ Step 3: Structuring test data to match the training feature grid...

================ 🏆 PROCESSING COMPLETE ================
💾 Transformed testing dataset saved safely at:
   /content/drive/MyDrive/Hackathon_05 06/super_dataset_C_test.csv
📊 Final Test Dimensions: 4749 samples | 65 features (including respondent_id)
